# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os
import gc
import json
import wandb
from tqdm.auto import tqdm
import warnings
from qlora_finetune_script import create_datasets, make_voxtral_collate_fn, get_prepared_model

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_finetune.ipynb"
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(model_id)
# Right padding for training
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.unk_token
    processor.tokenizer.pad_token_id = processor.tokenizer.unk_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)

# model = VoxtralForConditionalGeneration.from_pretrained(
#             model_id,
#             quantization_config=bnb_config,
#             attn_implementation="flash_attention_2",
#             device_map=device
#         )
# print(model)
# del model
# gc.collect()
# torch.cuda.empty_cache()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.
wandb: Currently logged in as: vitolus (vitolus-universit-ca-foscari-venezia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


#### Dataset Formatting for Multimodal SFT

In [2]:
df = pd.read_csv("./data/combined_multimodal_dataset_train.csv")
# Extract all unique user commands
unique_commands = df['User_Command'].unique().tolist()
print(f"Total Unique Commands: {len(unique_commands)}")

Total Unique Commands: 10184


#### Hyperpatameters tuning

In [ ]:
def create_sweep_func(train_dataset, eval_dataset):
    def train_sweep():
        gc.collect()
        torch.cuda.empty_cache()
        wandb.init()
        config = wandb.config
        # Reload Base Model (clears old adapters from memory)
        model = get_prepared_model(model_id, bnb_config, device, compute_dtype, processor)

        # Dynamic LoRA Config from Sweep
        lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        # Dynamic Training Args
        training_args = SFTConfig(
            output_dir="./models/voxtral-sweep",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=4,
            eval_strategy="steps",
            eval_steps=25,
            save_strategy="no",
            load_best_model_at_end=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            gradient_accumulation_steps=4,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            dataloader_prefetch_factor=2,
            learning_rate=config.learning_rate,
            max_steps=100,
            logging_steps=10,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            remove_unused_columns=False,
            dataset_kwargs={"skip_prepare_dataset": True},
            report_to="wandb",
            # loss_type="chunked_nll",
            neftune_noise_alpha=5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            weight_decay=0.01
        )
        # Initialize Trainer
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=make_voxtral_collate_fn(processor),
            processing_class=processor,
            peft_config=lora_config,
        )

        try:
            trainer.train()
        finally:
            del model, trainer
            gc.collect()
            torch.cuda.empty_cache()
    return train_sweep

HYPERTUNE = False # Set to True to run hyperparameter sweep, False for single training run
if HYPERTUNE:
    subset_commands = unique_commands[:1000]
    train_dataset, eval_dataset = create_datasets(df, subset_commands, processor)
    eval_dataset = eval_dataset.select(range(300))
    gc.collect()
    torch.cuda.empty_cache()
    project = "Voxtral-GLaDOS-Multimodal"

    sweep_config = {
        'method': 'bayes', # Bayesian optimization (smarter than random search)
        'metric': {'name': 'eval/loss', 'goal': 'minimize'},
        'early_terminate': {
            'type': 'hyperband',
            'min_iter': 3, # Minimum number of iterations to run
            'eta': 2 # Aggressiveness of early stopping (higher = more aggressive). Hyperband will stop poorly performing runs early based on intermediate results, allowing more resources for promising configurations.
        },
        'parameters': {
            'learning_rate': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 5e-4},
            'lora_r': {'values': [8, 16, 32]}, # Rank of the adapters
            'lora_alpha': {'values': [16, 32, 64]}, # Scaling factor
            'lora_dropout': {'values': [0.05, 0.1]} # Dropout for regularization
        }
    }
    api = wandb.Api()
    entity = api.default_entity or wandb.setup().settings.entity
    sweep_id = wandb.sweep(sweep_config, project=project)
    # Execute the sweep (Run 5 different combinations)
    wandb.agent(sweep_id, create_sweep_func(train_dataset, eval_dataset), count=5)
    print("\nSweep complete! Fetching the best parameters from W&B cloud...")
    best_params = api.sweep(f"{entity}/{project}/{sweep_id}").best_run().config
    print(f"Best Parameters Found: {best_params}")
    with open("./models/best_sweep_params.json", "w") as f:
        json.dump(best_params, f, indent=4)
    del train_dataset, eval_dataset, api, entity, sweep_id, best_params
    gc.collect()

#### Supervised Fine-Tuning with TRL's SFTTrainer

In [3]:
best_params = {
    'learning_rate': 2e-4,
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.1
    }
params_path = "./models/best_sweep_params.json"
if os.path.exists(params_path):
    print(f"Found existing configuration at {params_path}")
    with open(params_path, "r") as f:
        best_params = json.load(f)
print("Generating complete datasets for final fine-tuning...")
train_dataset, eval_dataset = create_datasets(df, unique_commands, processor)
del df, unique_commands
gc.collect()
torch.cuda.empty_cache()
print(f"Loading {model_id} in 4 bit precision...")
output_dir = "./models/voxtral-glados-sft"
run_id_file = os.path.join(output_dir, "wandb_run_id.txt")
last_checkpoint = None
wandb_run_id = None
if os.path.exists(output_dir):
    last_checkpoint = get_last_checkpoint(output_dir)
    if last_checkpoint and os.path.exists(run_id_file):
        with open(run_id_file, "r") as f:
            wandb_run_id = f.read().strip()

if last_checkpoint and wandb_run_id:
    print(f"Resuming W&B run: {wandb_run_id}...")
    wandb.init(project="Voxtral-GLaDOS-Multimodal", id=wandb_run_id, resume="must")
else:
    print("Starting a new W&B run...")
    wandb_run_id = wandb.util.generate_id()
    os.makedirs(output_dir, exist_ok=True)
    with open(run_id_file, "w") as f:
        f.write(wandb_run_id)
    wandb.init(project="Voxtral-GLaDOS-Multimodal", name="voxtral-GLaDOS", id=wandb_run_id, resume="allow")

model = get_prepared_model(model_id, bnb_config, device, compute_dtype, processor)

lora_config = LoraConfig(
    r=best_params['lora_r'],
    lora_alpha=best_params['lora_alpha'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=best_params['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=800,
    save_strategy="steps",
    save_steps=800,
    save_total_limit=3,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=8,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    learning_rate=best_params['learning_rate'],
    logging_steps=10,
    num_train_epochs=5,
    optim="paged_adamw_8bit", # paged_adamw_8bit use ram if vram is saturated (paging)
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False, # Crucial so the collator receives the dicts
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="wandb",
    # loss_type="chunked_nll", # Accumulate loss in smaller chunks to prevent overflow with long sequences
    neftune_noise_alpha=5, # Add a small amount of noise to the activations during training to improve generalization
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=make_voxtral_collate_fn(processor),
    processing_class=processor,
    peft_config=lora_config
)
trainer.model.print_trainable_parameters()

Found existing configuration at ./models/best_sweep_params.json
Generating complete datasets for final fine-tuning...
Train rows: 36120 | Eval rows: 4176
Loading mistralai/Voxtral-Mini-3B-2507 in 4 bit precision...
Resuming W&B run: rqba4kmy...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 61,931,520 || all params: 4,738,202,624 || trainable%: 1.3071


In [ ]:
print("Initiating QLoRA Multimodal Alignment...")
try:
    if last_checkpoint is not None:
        print(f"Resuming training from {last_checkpoint}...")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        print("Starting a new training run...")
        trainer.train()
    # Save the final adapter weights
    trainer.save_model(os.path.join(output_dir, "final_adapters"))
    print("Training complete. Adapters saved.")
finally:
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    wandb.finish()

Initiating QLoRA Multimodal Alignment...
Resuming training from ./models/voxtral-glados-sft/checkpoint-800...


[transformers] Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


Step,Training Loss,Validation Loss
